#**Entrenamiento red neuronal para plano sagital**

In [1]:
# Instalamos la librería oficial de YOLO
!pip install -q ultralytics

# Verificamos que PyTorch esté detectando la GPU correctamente
import torch
if torch.cuda.is_available():
    print(f"¡Excelente! GPU detectada: {torch.cuda.get_device_name(0)}")
else:
    print("ERROR: No se detectó GPU.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 81.4 MB/s eta 0:00:00
¡Excelente! GPU detectada: NVIDIA L4


In [7]:
# Descomprimir archivo de base de datos
!unzip -q /content/dataset_sagital.zip -d /content/

Configurar la estructura del archivo data.yaml para tensorflow:

In [9]:
import yaml

# Estructura de puntos (pi1, pi2, p3) = 3 keypoints
data = {
    'path': '/content/dataset_sagital', # Ruta base absoluta en Colab
    'train': 'images/train',
    'val': 'images/val',
    'names': {0: 'arco'},
    'kpt_shape': [3, 3] # 3 keypoints, 3 dimensiones (x, y, visibilidad)
}

# Guardamos el archivo
with open('/content/data.yaml', 'w') as f:
    yaml.dump(data, f)

print("Archivo data.yaml creado correctamente.")

Archivo data.yaml creado correctamente.


Reconstrucción de imagenes previo a entrenamiento. Tener en cuenta descpues hacer esto de manera local para la validación.

In [10]:
import os
import glob
from PIL import Image, ImageFile

# Obligamos a PIL a leer los archivos aunque les falte el marcador final
ImageFile.LOAD_TRUNCATED_IMAGES = True

# Buscamos absolutamente todas las imágenes en tu dataset de Colab
dataset_dir = '/content/dataset_sagital/images'
imagenes = glob.glob(f"{dataset_dir}/**/*.png", recursive=True) + glob.glob(f"{dataset_dir}/**/*.jpg", recursive=True)

print(f"Iniciando reconstrucción de {len(imagenes)} imágenes...")
reparadas = 0

for img_path in imagenes:
    try:
        # Abrimos la imagen (PIL ignora el error gracias a la variable de arriba)
        img = Image.open(img_path)
        img.load() # Forzamos la decodificación completa en RAM

        # Al guardarla sobre sí misma, Python escribe un archivo 100% sano desde cero
        img.save(img_path)
        reparadas += 1
    except Exception as e:
        print(f"Error en {img_path}: {e}")

print(f"Se reescribieron con éxito {reparadas} imágenes.")

Iniciando reconstrucción de 3381 imágenes...
Se reescribieron con éxito 3381 imágenes.


###**Entrenamiento**
* Modelo: Yolov8m
* Epocas: 50
* Batch: 8

In [ ]:
!yolo pose train data=/content/data.yaml model=yolov8m-pose.pt epochs=50 imgsz=1024 batch=8 patience=30 project=biomecanica_sagital name=modelo_sagital_unificado device=0 fliplr=0.0

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.76 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int